# Transaction Foundation Model on Ray — Part 7: Online serving with Ray Serve

<div align="left">
  <a target="_blank" href="https://console.anyscale.com/template-preview/fintech_transaction_fm"><img src="https://img.shields.io/badge/🚀 Run_on-Anyscale-9hf"></a>&nbsp;
  <a href="https://github.com/anyscale/templates/tree/main/templates/fintech_transaction_fm" role="button"><img src="https://img.shields.io/static/v1?label=&message=View%20On%20GitHub&color=586069&logo=github&labelColor=2f363d"></a>&nbsp;
</div>

**⏱️ Time to complete**: ~5 min

---

Part 5 produced embeddings in batch — the default production path (precompute on a schedule, push to a feature store, look up in milliseconds at authorization time). But fraud also needs a *real-time* path for cards whose latest activity isn't in the last batch yet. Here we put the encoder behind an HTTP endpoint with Ray Serve that returns an embedding and a fraud score in one call.

In [ ]:
import sys, os, time

DEMO_ROOT = os.path.abspath(os.getcwd())
if DEMO_ROOT not in sys.path:
    sys.path.insert(0, DEMO_ROOT)

import requests

import ray
from ray import serve
ray.init(ignore_reinit_error=True, runtime_env={"working_dir": DEMO_ROOT})

from src.paths import artifact_paths, get_demo_base_dir

SCALE = "mini"                       # same knob as the earlier parts
paths = artifact_paths(get_demo_base_dir(), SCALE)

## Serve the encoder behind an autoscaling endpoint

`TransactionEmbeddingService` (in `src/serve.py`) is a Ray Serve deployment: a `@serve.deployment` class that loads the model once per replica and scores a request in `__call__`. Its `autoscaling_config` runs **1–4 replicas**, adding them as concurrent load rises and scaling back to one when idle — Serve handles the replica lifecycle, the HTTP proxy, and load balancing.

It mirrors the two-tier pattern real shops use (e.g. Revolut's PRAGMA: a small model online, a larger one in batch). The **static** card-level fields change rarely, so the service caches them per card and runs the transformer only over the recent **dynamic** window — the same static/dynamic split from Part 3, now paying off as a latency optimization on the hot path.

`serve.run` deploys the app; we then POST one realistic request (built from a tokenized window) and read back the embedding and score.

In [ ]:
from src.serve import build_app
from src.utils import sample_serve_payload

serve.run(build_app(paths["checkpoint"]), name="txn-fm")

payload = sample_serve_payload(paths["tokenized_eval"])   # one card's window as a JSON request
t0 = time.perf_counter()
resp = requests.post("http://localhost:8000/", json=payload, timeout=30).json()
latency_ms = (time.perf_counter() - t0) * 1000

print(f"card_id     : {resp['card_id']}")
print(f"embedding   : {[round(x, 3) for x in resp['embedding'][:6]]} …  (dim {len(resp['embedding'])})")
print(f"fraud_score : {resp['fraud_score']:.4f}")
print(f"latency     : {latency_ms:.1f} ms  (single request, includes HTTP round-trip)")

serve.shutdown()

One honest caveat: the `fraud_score` here is a placeholder (a norm-based proxy on the embedding) so the endpoint is self-contained. In production you'd load the XGBoost head trained in Part 6 and return its probability — the embedding the service computes is exactly the `fm` feature that head expects, so it drops in directly.

---

## Next

**Part 8 — Run the whole pipeline on Anyscale**: the same seven stages as one headless command (`run_pipeline.py`), then submitted as a scheduled, autoscaling Anyscale Job — plus observability and fault tolerance.